# Actividad práctica — Worker pool en Go

1. Implementar un worker pool con **4 workers**
2. Enviar **20 jobs** por un channel
3. Recolectar resultados
4. Agregar `context.WithTimeout`
5. Ejecutar con `-race` y provocar una carrera para verla
6. **Pregunta de cierre:** ¿por qué Go puede tener miles de goroutines sin crear miles de threads del SO?

In [ ]:
%%bash
set -e
if [ ! -d /usr/local/go ]; then
  wget -q https://go.dev/dl/go1.22.5.linux-amd64.tar.gz -O /tmp/go.tgz
  tar -C /usr/local -xzf /tmp/go.tgz
fi
/usr/local/go/bin/go version

go version go1.22.5 linux/amd64


In [ ]:
import os
os.environ["PATH"] += ":/usr/local/go/bin"
os.environ["CGO_ENABLED"] = "1"   # necesario para el detector de carreras (-race)
os.environ["GOCACHE"] = "/content/.gocache"

In [ ]:
%%bash
mkdir -p /content/go_ej2
cd /content/go_ej2
mkdir -p 01_pool 02_timeout 03_carrera 04_bonus
[ -f go.mod ] || go mod init ejercicio2

go: creating new go.mod: module ejercicio2
go: to add module requirements and sums:
	go mod tidy


## 1–3) Worker pool (4 workers, 20 jobs, resultados recolectados)
Cada job calcula el factorial de `n` simulando ~100 ms de trabajo. Como los resultados llegan en cualquier orden, los guardamos y los **ordenamos por ID** al final.

In [ ]:
%%writefile /content/go_ej2/01_pool/main.go
package main

import (
	"fmt"
	"sort"
	"sync"
	"time"
)

const (
	numWorkers = 4
	numJobs    = 20
)

type Job struct {
	ID int
	N  int
}

type Result struct {
	JobID  int
	Worker int
	N      int
	Value  uint64
}

func factorial(n int) uint64 {
	r := uint64(1)
	for i := 2; i <= n; i++ {
		r *= uint64(i)
	}
	return r
}

func worker(id int, jobs <-chan Job, results chan<- Result, wg *sync.WaitGroup) {
	defer wg.Done()
	for j := range jobs {
		time.Sleep(100 * time.Millisecond) // simula trabajo
		results <- Result{JobID: j.ID, Worker: id, N: j.N, Value: factorial(j.N)}
	}
}

func main() {
	jobs := make(chan Job, numJobs)
	results := make(chan Result, numJobs)

	var wg sync.WaitGroup
	for w := 1; w <= numWorkers; w++ {
		wg.Add(1)
		go worker(w, jobs, results, &wg)
	}

	// Enviar 20 jobs y cerrar el channel (los workers salen del range)
	inicio := time.Now()
	for i := 1; i <= numJobs; i++ {
		jobs <- Job{ID: i, N: i}
	}
	close(jobs)

	// Cuando terminan todos los workers, cerramos results
	go func() {
		wg.Wait()
		close(results)
	}()

	// Recolectar
	var todos []Result
	for r := range results {
		todos = append(todos, r)
	}
	sort.Slice(todos, func(a, b int) bool { return todos[a].JobID < todos[b].JobID })

	porWorker := map[int]int{}
	for _, r := range todos {
		fmt.Printf("job %2d  (worker %d)  %2d! = %d\n", r.JobID, r.Worker, r.N, r.Value)
		porWorker[r.Worker]++
	}
	fmt.Println("\njobs por worker:", porWorker)
	fmt.Printf("total: %d resultados en %v (secuencial hubiera tardado ~2s)\n",
		len(todos), time.Since(inicio).Round(10*time.Millisecond))
}

Writing /content/go_ej2/01_pool/main.go


In [ ]:
%%bash
cd /content/go_ej2
go run ./01_pool 2>&1 || echo "--> terminó con error (código $?)"

job  1  (worker 1)   1! = 1
job  2  (worker 3)   2! = 2
job  3  (worker 4)   3! = 6
job  4  (worker 2)   4! = 24
job  5  (worker 3)   5! = 120
job  6  (worker 2)   6! = 720
job  7  (worker 1)   7! = 5040
job  8  (worker 4)   8! = 40320
job  9  (worker 4)   9! = 362880
job 10  (worker 3)  10! = 3628800
job 11  (worker 1)  11! = 39916800
job 12  (worker 2)  12! = 479001600
job 13  (worker 3)  13! = 6227020800
job 14  (worker 4)  14! = 87178291200
job 15  (worker 2)  15! = 1307674368000
job 16  (worker 1)  16! = 20922789888000
job 17  (worker 4)  17! = 355687428096000
job 18  (worker 3)  18! = 6402373705728000
job 19  (worker 1)  19! = 121645100408832000
job 20  (worker 2)  20! = 2432902008176640000

jobs por worker: map[1:5 2:5 3:5 4:5]
total: 20 resultados en 500ms (secuencial hubiera tardado ~2s)


## 4) Worker pool con `context.WithTimeout`
Ahora cada job tarda 300 ms. Con 4 workers y 20 jobs harían falta ~1.5 s, pero el timeout es de **1 s**, así que solo se completan unos ~12 jobs: el resto se cancela. Los workers chequean `ctx.Done()` tanto al tomar un job como mientras "trabajan".

In [ ]:
%%writefile /content/go_ej2/02_timeout/main.go
package main

import (
	"context"
	"fmt"
	"sync"
	"time"
)

type Job struct{ ID int }

type Result struct {
	JobID  int
	Worker int
	Err    error
}

func worker(ctx context.Context, id int, jobs <-chan Job, results chan<- Result, wg *sync.WaitGroup) {
	defer wg.Done()
	for {
		select {
		case <-ctx.Done():
			return
		case j, ok := <-jobs:
			if !ok {
				return // no hay más jobs
			}
			select {
			case <-time.After(300 * time.Millisecond): // trabajo
				results <- Result{JobID: j.ID, Worker: id}
			case <-ctx.Done():
				results <- Result{JobID: j.ID, Worker: id, Err: ctx.Err()}
				return
			}
		}
	}
}

func main() {
	ctx, cancel := context.WithTimeout(context.Background(), 1*time.Second)
	defer cancel()

	jobs := make(chan Job)
	results := make(chan Result)

	var wg sync.WaitGroup
	for w := 1; w <= 4; w++ {
		wg.Add(1)
		go worker(ctx, w, jobs, results, &wg)
	}
	go func() { wg.Wait(); close(results) }()

	// Productor que respeta la cancelación
	go func() {
		defer close(jobs)
		for i := 1; i <= 20; i++ {
			select {
			case jobs <- Job{ID: i}:
			case <-ctx.Done():
				return
			}
		}
	}()

	ok, canceladas := 0, 0
	for r := range results {
		if r.Err != nil {
			canceladas++
			fmt.Printf("job %2d  worker %d  %v\n", r.JobID, r.Worker, r.Err)
		} else {
			ok++
			fmt.Printf("job %2d  worker %d \n", r.JobID, r.Worker)
		}
	}
	fmt.Printf("\ncompletados: %d | cancelados a mitad de trabajo: %d | nunca iniciados: %d\n",
		ok, canceladas, 20-ok-canceladas)
	fmt.Println("motivo de cancelación:", ctx.Err())
}

Overwriting /content/go_ej2/02_timeout/main.go


In [ ]:
%%bash
cd /content/go_ej2
go run ./02_timeout 2>&1 || echo "--> terminó con error (código $?)"

job  2  worker 3 
job  1  worker 1 
job  4  worker 4 
job  3  worker 2 
job  8  worker 4 
job  6  worker 2 
job  5  worker 3 
job  7  worker 1 
job  9  worker 4 
job 12  worker 3 
job 10  worker 1 
job 11  worker 2 
job 13  worker 4  context deadline exceeded
job 16  worker 1  context deadline exceeded
job 15  worker 3  context deadline exceeded
job 14  worker 2  context deadline exceeded

completados: 12 | cancelados a mitad de trabajo: 4 | nunca iniciados: 4
motivo de cancelación: context deadline exceeded


## 5) Provocar una carrera y verla con `-race`
Los workers incrementan un contador compartido `procesados` **sin protección**. Es el error clásico: el programa "parece" funcionar, pero `-race` lo detecta.

In [ ]:
%%writefile /content/go_ej2/03_carrera/main.go
package main

import (
	"fmt"
	"sync"
)

func main() {
	jobs := make(chan int, 20)
	var wg sync.WaitGroup
	procesados := 0 // variable compartida

	for w := 1; w <= 4; w++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			for range jobs {
				procesados++
			}
		}()
	}

	for i := 1; i <= 20; i++ {
		jobs <- i
	}
	close(jobs)
	wg.Wait()

	fmt.Println("procesados (esperado 20):", procesados)
}

Overwriting /content/go_ej2/03_carrera/main.go


In [ ]:
%%bash
cd /content/go_ej2
go run -race ./03_carrera 2>&1 || echo "--> terminó con error (código $?)"

procesados (esperado 20): 20


La salida muestra `WARNING: DATA RACE` con dos stacks: *Write at ... by goroutine X* y *Previous write at ... by goroutine Y*, ambos en la línea `procesados++`.

### Corrección con `sync/atomic`

In [ ]:
%%writefile /content/go_ej2/03_carrera/main.go
package main

import (
	"fmt"
	"sync"
	"sync/atomic"
)

func main() {
	jobs := make(chan int, 20)
	var wg sync.WaitGroup
	var procesados atomic.Int64 // ✅ operaciones atómicas

	for w := 1; w <= 4; w++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			for range jobs {
				procesados.Add(1)
			}
		}()
	}

	for i := 1; i <= 20; i++ {
		jobs <- i
	}
	close(jobs)
	wg.Wait()

	fmt.Println("procesados (esperado 20):", procesados.Load())
}

Overwriting /content/go_ej2/03_carrera/main.go


In [ ]:
%%bash
cd /content/go_ej2
go run -race ./03_carrera 2>&1 || echo "--> terminó con error (código $?)"

Read at 0x00c00009e028 by goroutine 8:
  main.main.func1()
      /content/go_ej2/03_carrera/main.go:18 +0xad

Previous write at 0x00c00009e028 by goroutine 7:
  main.main.func1()
      /content/go_ej2/03_carrera/main.go:18 +0xc4

Goroutine 8 (running) created at:
  main.main()
      /content/go_ej2/03_carrera/main.go:15 +0xa4

Goroutine 7 (running) created at:
  main.main()
      /content/go_ej2/03_carrera/main.go:15 +0xa4
Write at 0x00c00009e028 by goroutine 8:
  main.main.func1()
      /content/go_ej2/03_carrera/main.go:18 +0xc4

Previous write at 0x00c00009e028 by goroutine 9:
  main.main.func1()
      /content/go_ej2/03_carrera/main.go:18 +0xc4

Goroutine 8 (running) created at:
  main.main()
      /content/go_ej2/03_carrera/main.go:15 +0xa4

Goroutine 9 (finished) created at:
  main.main()
      /content/go_ej2/03_carrera/main.go:15 +0xa4
procesados (esperado 20): 20
Found 2 data race(s)
exit status 66
--> terminó con error (código 1)



## Pregunta de cierre: ¿por qué Go puede tener miles de goroutines sin crear miles de threads del SO?

Porque **las goroutines no son threads del SO**: son tareas livianas que el *runtime de Go* planifica por su cuenta sobre un número chico de threads reales. Es un modelo **M:N** (muchas goroutines sobre pocos threads).

- **Scheduler propio (G-M-P):** cada **G** (goroutine) se ejecuta sobre un **M** (thread del SO) que necesita un **P** (procesador lógico). Hay tantos P como `GOMAXPROCS` (por defecto, la cantidad de núcleos). Miles de G se reparten entre unos pocos M.
- **Stacks diminutos y dinámicos:** una goroutine arranca con ~2 KB de stack que crece y se achica según haga falta. Un thread del SO reserva típicamente ~1–8 MB fijos. 100 000 goroutines ≈ cientos de MB como mucho; 100 000 threads serían inviables.
- **Cambio de contexto barato:** cambiar de goroutine ocurre en espacio de usuario (guardar unos pocos registros), sin pasar por el kernel, mucho más rápido que un context switch de threads.
- **Bloqueo cooperativo con el runtime:** cuando una goroutine espera un channel, un mutex, `time.Sleep` o I/O de red (vía *netpoller*), el runtime la **estaciona** y usa ese mismo thread para correr otra goroutine. Solo las syscalls realmente bloqueantes consumen un thread extra, y el runtime lo gestiona.

**Resumen:** 1000 goroutines esperando un channel ocupan memoria (unos pocos KB cada una), pero no 1000 threads; el SO solo ve unos pocos threads trabajando.

### Extra: demostrar lo que se planteamos antes
10.000 goroutines vivas a la vez, y cuántos threads del SO usa el proceso (leído de `/proc/self/status`).

In [ ]:
%%writefile /content/go_ej2/04_bonus/main.go
package main

import (
	"fmt"
	"os"
	"runtime"
	"strings"
	"sync"
)

func threadsSO() string {
	b, _ := os.ReadFile("/proc/self/status")
	for _, l := range strings.Split(string(b), "\n") {
		if strings.HasPrefix(l, "Threads:") {
			return strings.TrimSpace(strings.TrimPrefix(l, "Threads:"))
		}
	}
	return "?"
}

func main() {
	var wg sync.WaitGroup
	bloqueo := make(chan struct{}) // todas esperan acá
	for i := 0; i < 10000; i++ {
		wg.Add(1)
		go func() { defer wg.Done(); <-bloqueo }()
	}
	runtime.Gosched()

	fmt.Println("goroutines vivas:", runtime.NumGoroutine())
	fmt.Println("GOMAXPROCS:      ", runtime.GOMAXPROCS(0))
	fmt.Println("threads del SO:  ", threadsSO())

	close(bloqueo)
	wg.Wait()
}

Writing /content/go_ej2/04_bonus/main.go


In [ ]:
%%bash
cd /content/go_ej2
go run ./04_bonus 2>&1 || echo "--> terminó con error (código $?)"

goroutines vivas: 10001
GOMAXPROCS:       2
threads del SO:   4
